In [ ]:
#  import python libariries 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
#  import models  from Sk-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [ ]:
#  data loading 
df=pd.read_csv("loan_approval.csv")
# df.info()  df.isnull().sum() 

In [ ]:
#  DATA CLEANING 

numerical_cols=df.select_dtypes(include=["float64"]).columns
categorical_cols=df.select_dtypes(include=["object"]).columns

# handling with missing values   
num_imp= SimpleImputer(strategy="mean")
df[numerical_cols]=num_imp.fit_transform(df[numerical_cols])

cat_imp= SimpleImputer(strategy="most_frequent")
df[categorical_cols]=cat_imp.fit_transform(df[categorical_cols])

# cleaned_data into csv
cleaned_data=df.to_csv("cleaned_data.csv")


In [ ]:
#  Exploratory Data Analysis
classes_count= df["Loan_Approved"].value_counts()
plt.pie(classes_count,labels=['No','Yes'], autopct='%1.1f%%')
plt.title(" Loan approved or not ")
# In our data load approved = 30% and loan not approved=70%
df.info()


In [ ]:
#  analyze the categories data

df.head()
''' 
gender_count= df['Gender'].value_counts()
gen_bar=sns.barplot(gender_count)
gen_bar.bar_label(gen_bar.containers[0])

'''

cat_columns = [
    "Education_Level",
    "Employment_Status",
    "Marital_Status",
    'Loan_Purpose',
    "Property_Area",
    "Employer_Category"
]

fig, axes = plt.subplots(2,3 , figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, cat_columns):
    count = df[col].value_counts()
    bars = sns.barplot(
        x=count.index,
        y=count.values,
        ax=ax
    )

    bars.bar_label(bars.containers[0])
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig("01_categ_features.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# After Analyzing the data we got

# loan approved = 30% and loan not approved=70%
#  Males 621 and Females are 379 
#  Males 621 and Females are 379 
#  employment Status 
      #  salaried 515       contract 213
      #  self employed 182  unepmloyed 90
#  Marital _ status 
   # Married : 643
   # single  : 357
# Propoerty Area 
  # Urban : 517 Rural: 294  Semiurban :189
#  Employment Category 
   # private 422  govt 202 
   #  MNC 144 Business 135 unemploymed 97


In [ ]:
#  Analyze the income
sns.histplot(
    data=df,
    x='Applicant_Income',
    bins=20,
) #  max income is 10000-11k

sns.histplot(
    data=df,
    x='Coapplicant_Income',
    bins=20,
) #  most similar pattern now we analyzed that if applicatnt income high then co applicant income also high


#  Outtliers detection - Box plot
sns.boxplot(
    data=df,
    x="Loan_Approved",
    y="Applicant_Income",

)


In [ ]:
fig,axes=plt.subplots(3,2)
sns.boxplot(ax=axes[0,0], data=df, x="Loan_Approved",y="Applicant_Income")
sns.boxplot(ax=axes[0,1], data=df, x="Loan_Approved",y="Credit_Score")
sns.boxplot(ax=axes[1,0], data=df, x="Loan_Approved",y="DTI_Ratio")
sns.boxplot(ax=axes[1,1], data=df, x="Loan_Approved",y="Savings")
sns.boxplot(ax=axes[2,0], data=df, x="Loan_Approved",y="Age")
sns.boxplot(ax=axes[2,1], data=df, x="Loan_Approved",y="Loan_Amount")

plt.tight_layout()
plt.savefig("check_for_outlier.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#  Credit score with loan approved
sns.histplot(
    data=df,
    x='Credit_Score',
    hue='Loan_Approved',
    bins=20,
    multiple='dodge',
   
) #  credit score high->high chances for getting loan

sns.histplot(
    data=df,
    x='Applicant_Income',
    hue='Loan_Approved',
    bins=20,
    multiple='dodge',
   
)

# Remove un necessary column Applicant id
df=df.drop('Applicant_ID',axis=1)
df.head()


In [ ]:
#  Encoding
#  using label encoder sklean function 
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
le=LabelEncoder()
#  Encode target labels with value between 0 and n_classes-1  used for oridional  order will be in data 
df['Education_Level']= le.fit_transform(df['Education_Level'])
df['Loan_Approved']=le.fit_transform(df['Loan_Approved'])

#  one hot encoder each category is equal nominal  no order
ohe = OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore')

cols=[
    "Employment_Status",
    "Marital_Status",
    'Loan_Purpose',
    "Property_Area",
    "Employer_Category",
     'Gender']
 
encoded= ohe.fit_transform(df[cols]) # return 2d array 

#  conver 2d array into Data Frame
encoded_df= pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index= df.index)

# concatinate
df=pd.concat([df.drop(columns=cols),encoded_df],axis=1)


In [ ]:
#  encoded data 
encoded_data= df.to_csv('encoded_data.csv')
df.head()

In [ ]:
#  Find Correlation heatmaps

num_cols= df.select_dtypes(include='number')
correl_matrix=num_cols.corr()

plt.figure(figsize=(15,8))

sns.heatmap(
    correl_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm'
)

plt.savefig("heatmap_image.png", dpi=300, bbox_inches="tight")

# num_cols.corr()['Loan_Approved'].sort_values(ascending=False)



In [ ]:
# Train test split and feature scalling

X=df.drop(columns='Loan_Approved' ,axis=1)
Y=df['Loan_Approved']

X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)


In [ ]:
# Scalarize the data
scalar= StandardScaler()
x_train_scaled=scalar.fit_transform(X_train)
x_test_scaled=scalar.transform(X_test)


In [ ]:
# Logestic Regression Model

log_model= LogisticRegression()
log_model.fit(x_train_scaled,y_train)

y_pred=log_model.predict(x_test_scaled)

# Evaluation Matrix 
print(f'---- Logestic Regression Model ----')
print(f' Confusiion Matrix :\n {confusion_matrix(y_test,y_pred)} ')
print(f' Precision Score: {precision_score(y_test,y_pred)} ')
print(f' Recall Score:  {recall_score(y_test,y_pred)} ')
print(f' Accuracy Score:  {accuracy_score(y_test,y_pred)} ')
print(f' F1  Score:  {f1_score(y_test,y_pred)} ')
  

In [ ]:
# KNN  Model
knn_model= KNeighborsClassifier(n_neighbors=11)
knn_model.fit(x_train_scaled,y_train)

y_pred=knn_model.predict(x_test_scaled)

# Evaluation Matrix 
print(f'---- KNN  Model ----')
print(f' Confusiion Matrix :\n {confusion_matrix(y_test,y_pred)} ')
print(f' Precision Score: {precision_score(y_test,y_pred)} ')
print(f' Recall Score:  {recall_score(y_test,y_pred)} ')
print(f' Accuracy Score:  {accuracy_score(y_test,y_pred)} ')
print(f' F1  Score:  {f1_score(y_test,y_pred)} ')
  

In [ ]:
# Naive Bayes Model

naive_model= GaussianNB()
naive_model.fit(x_train_scaled,y_train)

y_pred=naive_model.predict(x_test_scaled)

# Evaluation Matrix 
print(f'---- Naive Bayes Model ----')
print(f' Confusiion Matrix :\n {confusion_matrix(y_test,y_pred)} ')
print(f' Precision Score: {precision_score(y_test,y_pred)} ')
print(f' Recall Score:  {recall_score(y_test,y_pred)} ')
print(f' Accuracy Score:  {accuracy_score(y_test,y_pred)} ')
print(f' F1  Score:  {f1_score(y_test,y_pred)} ')
  

In [ ]:
#  Feature Engineering
# Add or Transform 

df['DTI_Ratio_sq']= df['DTI_Ratio']**2
df['Credit_Score_sq']= df['Credit_Score']**2
df['Applicat_Income_log']=np.log1p(df['Applicant_Income']) # only for skewed data

X=df.drop(columns=['Loan_Approved','Credit_Score','DTI_Ratio']) 
Y=df['Loan_Approved']

#  Train Test scalling
X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

#  Scaling
scalar= StandardScaler()
x_train_scaled=scalar.fit_transform(X_train)
x_test_scaled=scalar.transform(X_test)




In [ ]:
# Logestic Regression Model after feature Engineering

log_model= LogisticRegression()
log_model.fit(x_train_scaled,y_train)

y_pred=log_model.predict(x_test_scaled)

# Evaluation Matrix 
print(f'---- Logestic Regression Model ----')
print(f' Confusiion Matrix :\n {confusion_matrix(y_test,y_pred)} ')
print(f' Precision Score: {precision_score(y_test,y_pred)} ')
print(f' Recall Score:  {recall_score(y_test,y_pred)} ')
print(f' Accuracy Score:  {accuracy_score(y_test,y_pred)} ')
print(f' F1  Score:  {f1_score(y_test,y_pred)} ')
  

In [ ]:
# KNN  Model
knn_model= KNeighborsClassifier(n_neighbors=11)
knn_model.fit(x_train_scaled,y_train)

y_pred=knn_model.predict(x_test_scaled)

# Evaluation Matrix 
print(f'---- KNN  Model ----')
print(f' Confusiion Matrix :\n {confusion_matrix(y_test,y_pred)} ')
print(f' Precision Score: {precision_score(y_test,y_pred)} ')
print(f' Recall Score:  {recall_score(y_test,y_pred)} ')
print(f' Accuracy Score:  {accuracy_score(y_test,y_pred)} ')
print(f' F1  Score:  {f1_score(y_test,y_pred)} ')
  

In [ ]:
# Naive Bayes Model
naive_model= GaussianNB()
naive_model.fit(x_train_scaled,y_train)

y_pred=naive_model.predict(x_test_scaled)

# Evaluation Matrix 
print(f'---- Naive Bayes Model ----')
print(f' Confusiion Matrix :\n {confusion_matrix(y_test,y_pred)} ')
print(f' Precision Score: {precision_score(y_test,y_pred)} ')
print(f' Recall Score:  {recall_score(y_test,y_pred)} ')
print(f' Accuracy Score:  {accuracy_score(y_test,y_pred)} ')
print(f' F1  Score:  {f1_score(y_test,y_pred)} ')
  